<a href="https://colab.research.google.com/github/Miranita-ar/Skripsi-Gojek-App-Review/blob/main/Code/(B)_Skripsi_IndoBERT_(Split_Data-Class_Weight-Tokenisasi).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install & Import Library

In [ ]:
# =======================================================
# SEL 1 : INSTALL LIBRARY
# =======================================================

!pip install -q transformers datasets accelerate evaluate
!pip install -q scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00


In [ ]:
# =======================================================
# SEL 1a : IMPORT LIBRARY
# =======================================================

import os
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from datasets import Dataset
from transformers import AutoTokenizer

pd.set_option("display.max_columns", None)

# Mount Google Drive

In [ ]:
# =======================================================
# SEL 2 : MOUNT DRIVE & FOLDER
# =======================================================

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/Skripsi_IndoBERT"

!mkdir -p "{PROJECT_DIR}/data/split"
!mkdir -p "{PROJECT_DIR}/data/tokenized"
!mkdir -p "{PROJECT_DIR}/data/tokenizer"
!mkdir -p "{PROJECT_DIR}/data/class_weight"

!mkdir -p "{PROJECT_DIR}/analysis"
!mkdir -p "{PROJECT_DIR}/models"
!mkdir -p "{PROJECT_DIR}/results"
!mkdir -p "{PROJECT_DIR}/logs"

print("✅ Struktur folder berhasil dibuat")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Struktur folder berhasil dibuat


# Load Data Preproceseesd (Notebook A)

In [ ]:
# =======================================================
# SEL 4 : LOAD DATA PREPROCESSED
# =======================================================

DATA_URL = "https://raw.githubusercontent.com/Miranita-ar/Skripsi-Gojek-App-Review/refs/heads/main/Data/data_preprocessed_multiclass.csv"

df_raw = pd.read_csv(DATA_URL)

print("="*70)
print("DATA PREPROCESSED")
print("="*70)

print(f"\nJumlah Data : {len(df_raw):,}")

display(df_raw.head())

DATA PREPROCESSED

Jumlah Data : 19,794


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,label,content_lower_case,content_clean,content_normalisasi
0,5adef64a-687b-44cb-ae83-fccac29af5f8,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"saya sudah memakai aplikasi gojek 4 tahun,alha...",5,0,5.47.1,2026-02-03 00:57:36,NaN,NaN,5.47.1,2,"saya sudah memakai aplikasi gojek 4 tahun,alha...",saya sudah memakai aplikasi gojek 4 tahun alha...,saya sudah memakai aplikasi gojek 4 tahun alha...
1,ee4c68a5-56ba-4460-a302-8f6c1fe3b919,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,jarang ada diskon,5,0,5.48.2,2026-02-03 00:51:32,NaN,NaN,5.48.2,2,jarang ada diskon,jarang ada diskon,jarang ada diskon
2,a1487729-8a06-4fb9-a99e-762150ba7438,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"makin berat aja apk, muter mulu mau beli tiket...",1,0,5.39.1,2026-02-03 00:37:37,"Hai Kak @Now You See Me Sony Handoko, mohon ma...",2026-02-03 01:46:54,5.39.1,0,"makin berat aja apk, muter mulu mau beli tiket...",makin berat aja apk muter mulu mau beli tiket krl,makin berat aja aplikasi muter terus mau beli ...
3,37bb5ff1-22d2-49aa-afa9-4c0a88564a8e,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,aplikasi terbaik,5,0,5.47.1,2026-02-03 00:27:07,NaN,NaN,5.47.1,2,aplikasi terbaik,aplikasi terbaik,aplikasi terbaik
4,06afd47c-c6be-4839-9931-9d90b918b039,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Di kasih limit di gopay pinjam 200rb,pengajuan...",2,0,5.48.2,2026-02-03 00:24:42,"Hai Kak Wirijaf, mohon maaf atas ketidaknyaman...",2026-02-03 01:42:50,5.48.2,0,"di kasih limit di gopay pinjam 200rb,pengajuan...",di kasih limit di gopay pinjam 200rb pengajuan...,di kasih limit di gopay pinjam 200 ribu pengaj...


In [ ]:
df = df_raw[["content_normalisasi", "label"]].copy()

df = df.rename(
    columns={
        "content_normalisasi":"text"
    }
)

df["label"] = df["label"].astype(int)

df = df.dropna(subset=["text"])

df = df[
    df["text"].str.strip() != ""
]

# =======================================================
# SEL 5 : DISTRIBUSI LABEL AWAL
# =======================================================

label_dist = (
    df["label"]
    .value_counts()
    .sort_index()
)

label_percent = (
    df["label"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

summary_awal = pd.DataFrame({
    "Jumlah": label_dist,
    "Persentase (%)": label_percent.round(2)
})

print("=== DISTRIBUSI LABEL AWAL ===")

display(summary_awal)

=== DISTRIBUSI LABEL AWAL ===


,Jumlah,Persentase (%)
label,,
0,6122,30.93
1,757,3.82
2,12915,65.25


In [ ]:
# =======================================================
# SEL 5a : PERSIAPAN DATA TRAINING
# =======================================================

df = df_raw[["content_normalisasi", "label"]].copy()

df = df.rename(
    columns={
        "content_normalisasi":"text"
    }
)

df["label"] = df["label"].astype(int)

df = df.dropna(subset=["text"])

df = df[
    df["text"].str.strip() != ""
]

print(f"Jumlah data siap training : {len(df):,}")

display(df.head())

Jumlah data siap training : 19,794


,text,label
0,saya sudah memakai aplikasi gojek 4 tahun alha...,2
1,jarang ada diskon,2
2,makin berat aja aplikasi muter terus mau beli ...,0
3,aplikasi terbaik,2
4,di kasih limit di gopay pinjam 200 ribu pengaj...,0


# Pembagian Data

In [ ]:
# =======================================================
# SEL 6 : STRATIFIED SPLIT
# =======================================================

SEED = 42

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["label"]
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

print("="*70)
print("HASIL SPLIT")
print("="*70)

print(f"Train : {len(train_df):,}")
print(f"Valid : {len(valid_df):,}")
print(f"Test  : {len(test_df):,}")

HASIL SPLIT
Train : 15,835
Valid : 1,979
Test  : 1,980


In [ ]:
# =======================================================
# SEL 7 : DISTRIBUSI SPLIT
# =======================================================

def distribusi(df_input, nama):

    temp = pd.DataFrame()

    temp["Jumlah"] = (
        df_input["label"]
        .value_counts()
        .sort_index()
    )

    temp["Persentase"] = (
        df_input["label"]
        .value_counts(normalize=True)
        .sort_index()*100
    ).round(2)

    print(f"\n=== {nama} ===")

    display(temp)

    return temp

train_dist = distribusi(train_df, "TRAIN")
valid_dist = distribusi(valid_df, "VALID")
test_dist = distribusi(test_df, "TEST")


=== TRAIN ===


,Jumlah,Persentase
label,,
0,4897,30.93
1,606,3.83
2,10332,65.25



=== VALID ===


,Jumlah,Persentase
label,,
0,612,30.92
1,76,3.84
2,1291,65.23



=== TEST ===


,Jumlah,Persentase
label,,
0,613,30.96
1,75,3.79
2,1292,65.25


In [ ]:
# =======================================================
# SEL 8 : SIMPAN CSV SPLIT
# =======================================================

train_df.to_csv(
    f"{PROJECT_DIR}/data/split/train_multiclass.csv",
    index=False
)

valid_df.to_csv(
    f"{PROJECT_DIR}/data/split/valid_multiclass.csv",
    index=False
)

test_df.to_csv(
    f"{PROJECT_DIR}/data/split/test_multiclass.csv",
    index=False
)

print("✅ CSV split berhasil disimpan")

✅ CSV split berhasil disimpan


In [ ]:
# title
# =======================================================
# SEL 8 : SIMPAN DATA SPLIT
# =======================================================

train_df.to_csv(
    f"{DATA_DIR}/train_multiclass.csv",
    index=False
)

valid_df.to_csv(
    f"{DATA_DIR}/valid_multiclass.csv",
    index=False
)

test_df.to_csv(
    f"{DATA_DIR}/test_multiclass.csv",
    index=False
)

print("train_multiclass.csv tersimpan")
print("valid_multiclass.csv tersimpan")
print("test_multiclass.csv tersimpan")

train_multiclass.csv tersimpan
valid_multiclass.csv tersimpan
test_multiclass.csv tersimpan


# Hitung Class Weight

gunakan train set saja

In [ ]:
# =======================================================
# SEL 9 : HITUNG CLASS WEIGHT
# =======================================================

classes = np.unique(
    train_df["label"]
)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"]
)

cw_df = pd.DataFrame({
    "label": classes,
    "weight": weights
})

display(cw_df)

,label,weight
0,0,1.077871
1,1,8.710121
2,2,0.510872


In [ ]:
# =======================================================
# SEL 10 : SIMPAN CLASS WEIGHT
# =======================================================

cw_df.to_csv(
    f"{PROJECT_DIR}/data/class_weight/class_weight.csv",
    index=False
)

np.save(
    f"{PROJECT_DIR}/data/class_weight/class_weight.npy",
    weights
)

with open(
    f"{PROJECT_DIR}/data/class_weight/class_weight.json",
    "w"
) as f:

    json.dump(
        weights.tolist(),
        f
    )

print("✅ Class Weight berhasil disimpan")

✅ Class Weight berhasil disimpan


In [ ]:
# =======================================================
# SEL 11 : SIMPAN RINGKASAN SPLIT
# =======================================================

split_summary = pd.concat(
    {
        "train": train_summary["Jumlah"],
        "valid": valid_summary["Jumlah"],
        "test": test_summary["Jumlah"]
    },
    axis=1
)

display(split_summary)

split_summary.to_csv(
    f"{ANALYSIS_DIR}/split_summary.csv"
)

print("split_summary.csv tersimpan")

,train,valid,test
label,,,
0,4897,612,613
1,606,76,75
2,10332,1291,1292


split_summary.csv tersimpan


# Load Tokenizer IndoBERT

In [ ]:
# =======================================================
# SEL 11 : LOAD TOKENIZER
# =======================================================

tokenizer = AutoTokenizer.from_pretrained(
    "indobenchmark/indobert-base-p2"
)

print("✅ Tokenizer berhasil dimuat")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

✅ Tokenizer berhasil dimuat


In [ ]:
# =======================================================
# SEL 12 : TOKENISASI
# =======================================================

MAX_LENGTH = 128

def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)
test_dataset = Dataset.from_pandas(test_df)

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

valid_dataset = valid_dataset.map(
    tokenize_function,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True
)

print("✅ Tokenisasi selesai")

Map:   0%|          | 0/15835 [00:00<?, ? examples/s]

Map:   0%|          | 0/1979 [00:00<?, ? examples/s]

Map:   0%|          | 0/1980 [00:00<?, ? examples/s]

✅ Tokenisasi selesai


In [ ]:
# =======================================================
# SEL 13 : SIMPAN TOKENIZED DATASET
# =======================================================

train_dataset.save_to_disk(
    f"{PROJECT_DIR}/data/tokenized/train_dataset"
)

valid_dataset.save_to_disk(
    f"{PROJECT_DIR}/data/tokenized/valid_dataset"
)

test_dataset.save_to_disk(
    f"{PROJECT_DIR}/data/tokenized/test_dataset"
)

print("✅ Dataset tokenized tersimpan")

Saving the dataset (0/1 shards):   0%|          | 0/15835 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1979 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1980 [00:00<?, ? examples/s]

✅ Dataset tokenized tersimpan


In [ ]:
# =======================================================
# SEL 14 : SIMPAN TOKENIZER
# =======================================================

tokenizer.save_pretrained(
    f"{PROJECT_DIR}/data/tokenizer"
)

print("✅ Tokenizer tersimpan")

✅ Tokenizer tersimpan


# Simpan Hasil

In [ ]:
# =======================================================
# SEL 15 : SPLIT SUMMARY
# =======================================================

split_summary = pd.concat(
    {
        "train": train_dist["Jumlah"],
        "valid": valid_dist["Jumlah"],
        "test": test_dist["Jumlah"]
    },
    axis=1
)

split_summary.to_csv(
    f"{PROJECT_DIR}/analysis/split_summary.csv"
)

display(split_summary)

,train,valid,test
label,,,
0,4897,612,613
1,606,76,75
2,10332,1291,1292


In [ ]:
# =======================================================
# SEL 16 : DISTRIBUSI PERSENTASE
# =======================================================

split_distribution = pd.concat(
    {
        "train": train_dist["Persentase"],
        "valid": valid_dist["Persentase"],
        "test": test_dist["Persentase"]
    },
    axis=1
)

split_distribution.to_csv(
    f"{PROJECT_DIR}/analysis/split_distribution.csv"
)

display(split_distribution)

,train,valid,test
label,,,
0,30.93,30.92,30.96
1,3.83,3.84,3.79
2,65.25,65.23,65.25


In [ ]:
# =======================================================
# SEL 17 : VERIFIKASI FILE
# =======================================================

for root, dirs, files in os.walk(PROJECT_DIR):

    level = root.replace(PROJECT_DIR, '').count(os.sep)

    indent = ' ' * 4 * level

    print(f'{indent}{os.path.basename(root)}/')

    subindent = ' ' * 4 * (level + 1)

    for f in files:
        print(f'{subindent}{f}')

Skripsi_IndoBERT/
    data/
        data_preprocessed_multiclass.csv
        data_labeled_multiclass.csv
        rating_before.csv
        label_before.csv
        rating_after.csv
        label_after.csv
        train_data.csv
        test_data.csv
        val_data.csv
        class_weights.json
        class_weights.txt
        metrics.py
        custom_trainer.py
        train_multiclass.csv
        valid_multiclass.csv
        test_multiclass.csv
        train_dataset/
            dataset_info.json
            state.json
            data-00000-of-00001.arrow
        val_dataset/
            dataset_info.json
            data-00000-of-00001.arrow
            state.json
        test_dataset/
            state.json
            dataset_info.json
            data-00000-of-00001.arrow
        tokenizer/
            tokenizer_config.json
            tokenizer.json
        split/
            train_multiclass.csv
            valid_multiclass.csv
            test_multiclass.csv
        token

In [ ]:
from datasets import load_from_disk

test_dataset = load_from_disk(
    "/content/drive/MyDrive/Skripsi_IndoBERT/data/tokenized/test_dataset"
)

print(test_dataset.features)

print("\n")

print(test_dataset[0])

{'text': Value('string'), 'label': Value('int64'), '__index_level_0__': Value('int64'), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}


{'text': 'pelayanan oke buat drivernya maaf baru pertama pakai jadi masih bingung', 'label': 2, '__index_level_0__': 12481, 'input_ids': [2, 1753, 4595, 968, 8794, 57, 2727, 440, 736, 2468, 472, 419, 4018, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [ ]:
import os

MODEL_DIR = "/content/drive/MyDrive/Skripsi_IndoBERT/models/exp1_lr2e5_bs8_ep3"

print(os.listdir(MODEL_DIR))

[]


In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

# Path ke file test_multiclass.csv (sesuai struktur folder Anda)
file_path = '/content/drive/MyDrive/Skripsi_IndoBERT/data/split/test_multiclass.csv'

# Load file CSV
test_df = pd.read_csv(file_path)

# Lihat 5 data pertama
print(test_df.head())

# Cek jumlah data
print(f"Jumlah data test: {len(test_df)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
                                                text  label
0  pelayanan oke buat drivernya maaf baru pertama...      2
1                          3 kali dikecewakan driver      0
2  top banget deh pokoknya aku saranin kalian jug...      2
3         membantu sekali buat perjalanan jauh dekat      2
4                       sukses selalu bwt pra driver      2
Jumlah data test: 1980
